# 🧹 Limpieza de Datos - Cafe Sales Dataset

**Fuente:** [Kaggle - Cafe Sales Dirty Data for Cleaning Training](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training)  
**Autor:** Luciano Rivas  
**Fecha de inicio:** Marzo 2026  

---

## 📋 Objetivo del proyecto

Este dataset contiene registros de ventas de un café con errores intencionales:
valores faltantes, formatos incorrectos y datos inválidos.

**Objetivo de negocio:** Generar un dataset limpio y confiable que permita
analizar métricas de ventas por producto, ubicación y método de pago.

**Tecnologías:** Python · Pandas · Matplotlib · Seaborn


---
## 1. Configuración del entorno


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)


In [2]:
df = pd.read_csv("../data/raw/dirty_cafe_sales.csv")
print(f"Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()


Dimensiones del dataset: 10000 filas × 8 columnas


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


---
## 2. Exploración inicial (EDA)

Antes de limpiar cualquier dato, el primer paso es entender la estructura
del dataset y **mapear todos los problemas existentes**.


### 2.1 Tipos de datos y estructura general


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


> **📝 Observación:** Todas las columnas tienen tipo `object` (texto),
> incluyendo columnas que deberían ser numéricas (`Quantity`, `Price Per Unit`,
> `Total Spent`) y de fecha (`Transaction Date`).
> Esto indica que los valores inválidos impidieron la detección automática de tipos.


### 2.2 Valores faltantes y erróneos por columna


In [4]:
df.isna().sum().sort_values(ascending=False)

Location            3265
Payment Method      2579
Item                 333
Price Per Unit       179
Total Spent          173
Transaction Date     159
Quantity             138
Transaction ID         0
dtype: int64

In [5]:
print("=== LOCATION ===")
print(df['Location'].value_counts(dropna=False))
print("\n=== PAYMENT METHOD ===")
print(df['Payment Method'].value_counts(dropna=False))
print("\n=== ITEM ===")
print(df['Item'].value_counts(dropna=False))

=== LOCATION ===
Location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64

=== PAYMENT METHOD ===
Payment Method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

=== ITEM ===
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64


> **📝 Observación:**
> - `Location` tiene **3.961 datos desconocidos** (~40% del total):
>   3.265 NaN + 358 "ERROR" + 338 "UNKNOWN"
> - `Payment Method` tiene **3.178 datos desconocidos** (~32% del total)
> - Los valores "ERROR" y "UNKNOWN" deben ser tratados como NaN en el
>   próximo paso, ya que semánticamente significan ausencia de dato real

### 2.3 Duplicados

In [6]:
print(f"Filas completamente duplicadas: {df.duplicated().sum()}")
print(f"Transaction IDs duplicados:     {df.duplicated(subset=['Transaction ID']).sum()}")

Filas completamente duplicadas: 0
Transaction IDs duplicados:     0


### 2.4 Formato de la columna fecha


In [7]:
print("Muestra de valores en Transaction Date:")
print(df['Transaction Date'].head(10))
print(f"\nValores nulos en fecha: {df['Transaction Date'].isna().sum()}")

Muestra de valores en Transaction Date:
0    2023-09-08
1    2023-05-16
2    2023-07-19
3    2023-04-27
4    2023-06-11
5    2023-03-31
6    2023-10-06
7    2023-10-28
8    2023-07-28
9    2023-12-31
Name: Transaction Date, dtype: object

Valores nulos en fecha: 159


---
## ✅ Resumen de problemas detectados

| Columna | Tipo actual | Tipo correcto | Problemas encontrados |
|---|---|---|---|
| Transaction ID | object | object | ✅ Sin problemas |
| Item | object | object | NaN + "ERROR" + "UNKNOWN" |
| Quantity | object | int | NaN + "ERROR" + "UNKNOWN" |
| Price Per Unit | object | float | NaN + "ERROR" + "UNKNOWN" |
| Total Spent | object | float | NaN + "ERROR" + "UNKNOWN" |
| Payment Method | object | object | ~32% datos desconocidos |
| Location | object | object | ~40% datos desconocidos |
| Transaction Date | object | datetime | Algunos NaN, formato YYYY-MM-DD |

**Plan de limpieza:**
1. Unificar "ERROR", "UNKNOWN", "N/A" → `NaN`
2. Convertir tipos de datos (numéricos + fecha)
3. Imputar numéricos con la fórmula `Total Spent = Quantity × Price Per Unit`
4. Rellenar categóricas faltantes con `"Unknown"`
5. Eliminar filas sin `Total Spent`
6. Exportar dataset limpio

---
## 3. Limpieza de datos

### 3.1 Unificación de valores inválidos como NaN

Durante el EDA detectamos que varios valores textuales (`"ERROR"`, `"UNKNOWN"`, `"N/A"`) 
representan ausencia de dato real. Pandas no los reconoce automáticamente como NaN, 
por lo que deben ser reemplazados explícitamente.

In [8]:
# Lista de valores que representan dato faltante
valores_invalidos = ["ERROR", "UNKNOWN", "N/A", ""]

# Reemplazar en todo el DataFrame
df.replace(valores_invalidos, np.nan, inplace=True)

print("✅ Valores inválidos unificados como NaN")

✅ Valores inválidos unificados como NaN


#### Verificación del reemplazo

In [9]:
# Verificar que los valores inválidos desaparecieron
print("=== Valores faltantes por columna (después del reemplazo) ===\n")
print(df.isna().sum().sort_values(ascending=False))

print("\n=== Location - Top valores únicos ===")
print(df['Location'].value_counts(dropna=False).head())

print("\n=== Payment Method - Top valores únicos ===")
print(df['Payment Method'].value_counts(dropna=False).head())

=== Valores faltantes por columna (después del reemplazo) ===

Location            3961
Payment Method      3178
Item                 969
Price Per Unit       533
Total Spent          502
Quantity             479
Transaction Date     460
Transaction ID         0
dtype: int64

=== Location - Top valores únicos ===
Location
NaN         3961
Takeaway    3022
In-store    3017
Name: count, dtype: int64

=== Payment Method - Top valores únicos ===
Payment Method
NaN               3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64


> **📝 Resultado:**
> - Los valores `"ERROR"` y `"UNKNOWN"` ya no aparecen en los `value_counts()`
> - Los NaN en `Location` aumentaron de 3.265 a **3.961** (se sumaron los 696 valores inválidos)
> - Los NaN en `Payment Method` aumentaron de ~500 a **3.178** (se sumaron los valores inválidos)
> - El dataset ahora tiene todos los datos faltantes estandarizados como `NaN`, 
>   lo que facilita la siguiente etapa de conversión de tipos

---
### 3.2 Conversión de tipos de datos

Con los valores inválidos ya estandarizados como NaN, podemos convertir 
las columnas a sus tipos correctos. Pandas podrá interpretar correctamente 
los datos numéricos y de fecha sin errores.

#### Columnas numéricas

In [10]:
# Convertir columnas numéricas
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')

print("✅ Columnas numéricas convertidas a tipo float")

✅ Columnas numéricas convertidas a tipo float


#### Columna de fecha

In [11]:
# Convertir Transaction Date a tipo datetime
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

print("✅ Transaction Date convertida a tipo datetime64")

✅ Transaction Date convertida a tipo datetime64


#### Verificación de tipos

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              9031 non-null   object        
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    6822 non-null   object        
 6   Location          6039 non-null   object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 625.1+ KB


> **📝 Resultado:**
> - `Quantity`, `Price Per Unit` y `Total Spent` ahora son **float64**
> - `Transaction Date` ahora es **datetime64[ns]**
> - Las columnas categóricas (`Item`, `Location`, `Payment Method`) permanecen como **object**
> - Los valores que no pudieron convertirse se transformaron automáticamente en NaN 
>   gracias al parámetro `errors='coerce'`
> - **Estado actual:** Dataset con tipos correctos, listo para imputación numérica

In [13]:
# Vista previa del dataset con tipos corregidos
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.00,2.00,4.00,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.00,3.00,12.00,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.00,1.00,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.00,5.00,10.00,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2.00,2.00,4.00,Digital Wallet,In-store,2023-06-11
